# 13 Summary-guided document-level BioBART

This experiment tests whether a concise document summary improves document-level biomedical text simplification. It follows the CLEF 2025 SimpleText document-level setting described in [paper 344](https://ceur-ws.org/Vol-4038/paper_344.pdf) and adapts the two-stage summary-guided design described by [Marturi and Elwazzan (2025)](https://arxiv.org/abs/2508.11816).

1. Llama 3.1 8B generates a concise summary through Ollama.
2. The summary and original document guide the already fine-tuned document-level BioBART model from notebook 12.

BioBART is used only for inference here. If the trained model is missing, run notebook 12 first; this notebook does not retrain from scratch.

## 1. Setup and configuration

In [ ]:
from __future__ import annotations

import gc
import json
import os
import random
import sys
import urllib.error
import urllib.request
from pathlib import Path
from typing import Any

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

LOCAL_CACHE_DIR = PROJECT_ROOT / ".cache"
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(LOCAL_CACHE_DIR / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(LOCAL_CACHE_DIR))
os.environ.setdefault("HF_HOME", str(LOCAL_CACHE_DIR))
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

SEED = 42
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.1:8b")
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://127.0.0.1:11434")

DATA_PATH = PROJECT_ROOT / "data" / "document" / "cochraneauto_docs_test.csv"
RESULTS_DIR = PROJECT_ROOT / "results"
SUMMARY_PATH = RESULTS_DIR / "document_summaries.csv"
PREDICTION_PATH = RESULTS_DIR / "summary_guided_biobart_predictions.csv"
METRICS_PATH = RESULTS_DIR / "summary_guided_biobart_metrics.csv"
COMPARISON_PATH = RESULTS_DIR / "document_model_comparison.csv"

# Notebook 12 is expected to save both model and tokenizer here.
BIOBART_MODEL_DIR = PROJECT_ROOT / "models" / "biobart_document_level" / "best_model"
MAX_SOURCE_LENGTH = 768

SUMMARY_GENERATION_CONFIG = {
    "temperature": 0.2,
    "top_p": 0.9,
    "max_new_tokens": 256,
    "seed": SEED,
}

BIOBART_GENERATION_CONFIG = {
    "num_beams": 4,
    "length_penalty": 0.9,
    "no_repeat_ngram_size": 3,
    "early_stopping": True,
    "max_new_tokens": 512,
}

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Project root:", PROJECT_ROOT)
print("Device:", device)
print("Summary cache:", SUMMARY_PATH.relative_to(PROJECT_ROOT))
print("BioBART model directory:", BIOBART_MODEL_DIR.relative_to(PROJECT_ROOT))

## 2. Load document-level test data

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Document test data not found: {DATA_PATH}")

test_df = pd.read_csv(DATA_PATH, dtype={"pair_id": str})
required_columns = {"pair_id", "complex", "simple"}
missing_columns = sorted(required_columns - set(test_df.columns))
if missing_columns:
    raise ValueError(f"Document test data is missing columns: {missing_columns}")
if test_df["pair_id"].duplicated().any():
    duplicate_ids = test_df.loc[test_df["pair_id"].duplicated(), "pair_id"].tolist()
    raise ValueError(f"Duplicate pair_id values in document test data: {duplicate_ids[:10]}")

test_df = test_df[["pair_id", "complex", "simple"]].copy()
test_df["complex"] = test_df["complex"].fillna("").astype(str)
test_df["simple"] = test_df["simple"].fillna("").astype(str)

print("Test documents:", len(test_df))
print("Average source words:", round(test_df["complex"].str.split().str.len().mean(), 2))
print("Average reference words:", round(test_df["simple"].str.split().str.len().mean(), 2))

## 3. Stage 1 — concise summaries with Llama 3.1 8B

Summaries are cached in `results/document_summaries.csv`. Valid cached rows are reused; blank and failed rows are retried.

In [ ]:
SUMMARY_PROMPT_TEMPLATE = """You are an expert biomedical summarizer.

Summarize the following biomedical document.

Requirements:
- Preserve the main findings.
- Keep the population, intervention and conclusions.
- Remove methodological and statistical details whenever possible.
- Do not invent information.
- Produce a concise summary.

Document:

{complex_document}

Summary:
"""


def build_summary_prompt(complex_document: str) -> str:
    return SUMMARY_PROMPT_TEMPLATE.format(complex_document=str(complex_document).strip())


def ollama_request(path: str, payload: dict[str, Any] | None = None, timeout: int = 120) -> dict[str, Any]:
    url = f"{OLLAMA_URL}{path}"
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.URLError as exc:
        raise RuntimeError(
            f"Could not reach Ollama at {OLLAMA_URL}. Start Ollama and install {OLLAMA_MODEL}."
        ) from exc


def ensure_ollama_model(model_name: str) -> None:
    response = ollama_request("/api/tags", timeout=15)
    available_models = [model.get("name", "") for model in response.get("models", [])]
    is_available = model_name in available_models or any(
        name.startswith(f"{model_name}-") for name in available_models
    )
    if not is_available:
        available = ", ".join(available_models) if available_models else "no local models"
        raise RuntimeError(
            f"Ollama model {model_name} is not installed. Available: {available}. "
            f"Install it with: ollama pull {model_name}"
        )
    print("Using Ollama model:", model_name)


def clean_summary(text: str) -> str:
    summary = str(text).strip()
    for prefix in ["Summary:", "Concise summary:", "Here is the summary:"]:
        if summary.lower().startswith(prefix.lower()):
            summary = summary[len(prefix):].strip()
    return " ".join(summary.split())


def generate_summary(complex_document: str) -> str:
    response = ollama_request(
        "/api/generate",
        payload={
            "model": OLLAMA_MODEL,
            "prompt": build_summary_prompt(complex_document),
            "stream": False,
            "options": {
                "temperature": SUMMARY_GENERATION_CONFIG["temperature"],
                "top_p": SUMMARY_GENERATION_CONFIG["top_p"],
                "num_predict": SUMMARY_GENERATION_CONFIG["max_new_tokens"],
                "seed": SUMMARY_GENERATION_CONFIG["seed"],
            },
        },
        timeout=300,
    )
    summary = clean_summary(response.get("response", ""))
    return summary if summary else "[SUMMARY_FAILED]"

In [ ]:
INVALID_SUMMARIES = {"", "[SUMMARY_FAILED]"}


def generate_summaries(input_df: pd.DataFrame) -> pd.DataFrame:
    summary_df = input_df[["pair_id", "complex"]].copy()
    summary_df["summary"] = ""

    if SUMMARY_PATH.exists():
        cached_df = pd.read_csv(SUMMARY_PATH, dtype={"pair_id": str})
        if {"pair_id", "summary"}.issubset(cached_df.columns):
            cached_df["summary"] = cached_df["summary"].fillna("").astype(str).str.strip()
            cached_df = cached_df.drop_duplicates("pair_id", keep="last")
            cached_map = cached_df.set_index("pair_id")["summary"]
            summary_df["summary"] = summary_df["pair_id"].map(cached_map).fillna("")

    pending_mask = summary_df["summary"].isin(INVALID_SUMMARIES)
    pending_df = summary_df.loc[pending_mask].copy()
    print("Cached summaries reused:", len(summary_df) - len(pending_df))
    print("Summaries to generate:", len(pending_df))

    if not pending_df.empty:
        ensure_ollama_model(OLLAMA_MODEL)

    for completed, (idx, row) in enumerate(
        tqdm(pending_df.iterrows(), total=len(pending_df), desc="Summarizing documents"),
        start=1,
    ):
        try:
            summary = generate_summary(row["complex"])
        except Exception as exc:
            summary = "[SUMMARY_FAILED]"
            print(f"Summary failed for pair_id={row['pair_id']}: {exc}")
        summary_df.loc[idx, "summary"] = summary

        if completed % 10 == 0:
            summary_df.to_csv(SUMMARY_PATH, index=False)
            print(f"Saved summary checkpoint after {completed} new documents")

    summary_df.to_csv(SUMMARY_PATH, index=False)
    print("Saved summaries to:", SUMMARY_PATH.relative_to(PROJECT_ROOT))
    return summary_df


summary_df = generate_summaries(test_df)
invalid_summary_mask = summary_df["summary"].fillna("").astype(str).str.strip().isin(INVALID_SUMMARIES)
if invalid_summary_mask.any():
    invalid_ids = summary_df.loc[invalid_summary_mask, "pair_id"].tolist()
    raise RuntimeError(
        f"Stage 1 has {len(invalid_ids)} incomplete summaries. Rerun this cell. "
        f"Affected pair_id values: {invalid_ids[:10]}"
    )

experiment_df = test_df.merge(summary_df[["pair_id", "summary"]], on="pair_id", how="left", validate="one_to_one")
print(experiment_df[["pair_id", "summary"]].head(3).to_string(index=False))

## 4. Stage 2 — summary-guided BioBART inference

The model and tokenizer must come from the document-level checkpoint produced by notebook 12. No training occurs in this notebook.

In [ ]:
required_model_files = [BIOBART_MODEL_DIR / "config.json"]
if not BIOBART_MODEL_DIR.exists() or not all(path.exists() for path in required_model_files):
    raise FileNotFoundError(
        f"Fine-tuned document-level BioBART model not found at {BIOBART_MODEL_DIR}. "
        "Run notebook 12_document_biobart_finetuning.ipynb first. "
        "Notebook 13 intentionally does not retrain BioBART from scratch."
    )

tokenizer = AutoTokenizer.from_pretrained(str(BIOBART_MODEL_DIR), local_files_only=True)
biobart_model = AutoModelForSeq2SeqLM.from_pretrained(str(BIOBART_MODEL_DIR), local_files_only=True)
biobart_model.to(device)
biobart_model.eval()

model_position_limit = int(getattr(biobart_model.config, "max_position_embeddings", MAX_SOURCE_LENGTH))
effective_source_length = min(MAX_SOURCE_LENGTH, model_position_limit)
print("Loaded fine-tuned BioBART from:", BIOBART_MODEL_DIR.relative_to(PROJECT_ROOT))
print("Maximum encoder input length:", effective_source_length)

In [ ]:
BIOBART_INPUT_TEMPLATE = """Summary:

{generated_summary}

Original document:

{complex_document}

Simplify the original document for a general audience.
"""


def build_biobart_input(generated_summary: str, complex_document: str) -> str:
    return BIOBART_INPUT_TEMPLATE.format(
        generated_summary=str(generated_summary).strip(),
        complex_document=str(complex_document).strip(),
    )


def generate_simplification(complex_document: str, summary: str) -> str:
    model_input = build_biobart_input(summary, complex_document)
    encoded = tokenizer(
        model_input,
        return_tensors="pt",
        max_length=effective_source_length,
        truncation=True,
    ).to(device)
    with torch.inference_mode():
        generated_ids = biobart_model.generate(**encoded, **BIOBART_GENERATION_CONFIG)
    prediction = tokenizer.decode(generated_ids[0], skip_special_tokens=True).strip()
    return prediction if prediction else "[GENERATION_FAILED]"

In [ ]:
OUTPUT_COLUMNS = ["pair_id", "complex", "summary", "simple", "prediction"]
INVALID_PREDICTIONS = {"", "[GENERATION_FAILED]"}


def generate_predictions(input_df: pd.DataFrame) -> pd.DataFrame:
    prediction_df = input_df[["pair_id", "complex", "summary", "simple"]].copy()
    prediction_df["prediction"] = ""

    if PREDICTION_PATH.exists():
        cached_df = pd.read_csv(PREDICTION_PATH, dtype={"pair_id": str})
        if set(OUTPUT_COLUMNS).issubset(cached_df.columns):
            cached_df = cached_df.drop_duplicates("pair_id", keep="last").set_index("pair_id")
            for idx, row in prediction_df.iterrows():
                pair_id = row["pair_id"]
                if pair_id not in cached_df.index:
                    continue
                cached_row = cached_df.loc[pair_id]
                cached_prediction = str(cached_row.get("prediction", "")).strip()
                summary_matches = str(cached_row.get("summary", "")).strip() == str(row["summary"]).strip()
                source_matches = str(cached_row.get("complex", "")).strip() == str(row["complex"]).strip()
                if summary_matches and source_matches and cached_prediction not in INVALID_PREDICTIONS and cached_prediction.lower() != "nan":
                    prediction_df.loc[idx, "prediction"] = cached_prediction

    pending_df = prediction_df.loc[prediction_df["prediction"].isin(INVALID_PREDICTIONS)].copy()
    print("Cached predictions reused:", len(prediction_df) - len(pending_df))
    print("Predictions to generate:", len(pending_df))

    for completed, (idx, row) in enumerate(
        tqdm(pending_df.iterrows(), total=len(pending_df), desc="Summary-guided BioBART"),
        start=1,
    ):
        try:
            prediction = generate_simplification(row["complex"], row["summary"])
        except Exception as exc:
            prediction = "[GENERATION_FAILED]"
            print(f"Generation failed for pair_id={row['pair_id']}: {exc}")
        prediction_df.loc[idx, "prediction"] = prediction

        if completed % 10 == 0:
            prediction_df[OUTPUT_COLUMNS].to_csv(PREDICTION_PATH, index=False)
            print(f"Saved prediction checkpoint after {completed} new documents")

    prediction_df = prediction_df[OUTPUT_COLUMNS]
    prediction_df.to_csv(PREDICTION_PATH, index=False)
    print("Saved predictions to:", PREDICTION_PATH.relative_to(PROJECT_ROOT))
    return prediction_df


prediction_df = generate_predictions(experiment_df)
invalid_prediction_mask = prediction_df["prediction"].fillna("").astype(str).str.strip().isin(INVALID_PREDICTIONS)
if invalid_prediction_mask.any():
    invalid_ids = prediction_df.loc[invalid_prediction_mask, "pair_id"].tolist()
    raise RuntimeError(
        f"Stage 2 has {len(invalid_ids)} incomplete predictions. Rerun this cell. "
        f"Affected pair_id values: {invalid_ids[:10]}"
    )

## 5. Evaluation

In [ ]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import compute_metrics, save_metrics

metrics_summary = compute_metrics(prediction_df)

source_lengths = prediction_df["complex"].str.split().str.len().astype(float)
prediction_lengths = prediction_df["prediction"].str.split().str.len().astype(float)
summary_lengths = prediction_df["summary"].str.split().str.len().astype(float)
per_document_compression = prediction_lengths.div(source_lengths.replace(0, np.nan))

diagnostic_metrics = pd.DataFrame([
    {"metric": "Average compression ratio", "score": float(per_document_compression.mean())},
    {"metric": "Average prediction length", "score": float(prediction_lengths.mean())},
    {"metric": "Average summary length", "score": float(summary_lengths.mean())},
])
metrics_summary = pd.concat([metrics_summary, diagnostic_metrics], ignore_index=True)
save_metrics(metrics_summary, METRICS_PATH)

print(metrics_summary.to_string(index=False))
print("Saved metrics to:", METRICS_PATH.relative_to(PROJECT_ROOT))

## 6. Comparison with document-level baselines

In [ ]:
def metric_value(metrics_df: pd.DataFrame, metric_name: str) -> float:
    if not {"metric", "score"}.issubset(metrics_df.columns):
        return float("nan")
    matches = metrics_df.loc[metrics_df["metric"].eq(metric_name), "score"]
    return float(matches.iloc[0]) if not matches.empty else float("nan")


def load_baseline_metrics(model_name: str, prediction_path: Path, metrics_path: Path) -> dict[str, Any]:
    if metrics_path.exists():
        model_metrics = pd.read_csv(metrics_path)
    elif prediction_path.exists():
        baseline_df = pd.read_csv(prediction_path, dtype={"pair_id": str})
        if not {"complex", "simple", "prediction"}.issubset(baseline_df.columns):
            print(f"Skipping {model_name}: prediction file has an incompatible schema.")
            model_metrics = pd.DataFrame(columns=["metric", "score"])
        else:
            invalid = baseline_df["prediction"].fillna("").astype(str).str.strip().isin(
                ["", "[GENERATION_FAILED]", "[EMPTY_OUTPUT]"]
            )
            if invalid.any():
                print(f"Skipping {model_name}: {int(invalid.sum())} predictions are incomplete.")
                model_metrics = pd.DataFrame(columns=["metric", "score"])
            else:
                model_metrics = compute_metrics(baseline_df)
                save_metrics(model_metrics, metrics_path)
    else:
        print(f"Skipping {model_name}: no metrics or predictions found.")
        model_metrics = pd.DataFrame(columns=["metric", "score"])

    return {
        "Model": model_name,
        "SARI": metric_value(model_metrics, "SARI"),
        "BLEU": metric_value(model_metrics, "BLEU"),
        "BERTScore F1": metric_value(model_metrics, "BERTScore F1"),
    }


comparison_rows = [
    load_baseline_metrics(
        "Document Llama",
        RESULTS_DIR / "llama_document_level_predictions.csv",
        RESULTS_DIR / "llama_document_level_metrics.csv",
    ),
    load_baseline_metrics(
        "Document BioBART",
        RESULTS_DIR / "biobart_document_level_predictions.csv",
        RESULTS_DIR / "biobart_document_level_metrics.csv",
    ),
    {
        "Model": "Summary-guided BioBART",
        "SARI": metric_value(metrics_summary, "SARI"),
        "BLEU": metric_value(metrics_summary, "BLEU"),
        "BERTScore F1": metric_value(metrics_summary, "BERTScore F1"),
    },
]

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(COMPARISON_PATH, index=False)
print(comparison_df.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print("Saved comparison to:", COMPARISON_PATH.relative_to(PROJECT_ROOT))

## 7. Qualitative diagnostics

In [ ]:
diagnostic_examples = prediction_df.sample(n=min(5, len(prediction_df)), random_state=SEED)

for example_number, row in enumerate(diagnostic_examples.itertuples(index=False), start=1):
    print("=" * 100)
    print(f"Example {example_number} | pair_id={row.pair_id}")
    print("\nOriginal document:\n")
    print(row.complex)
    print("\nGenerated summary:\n")
    print(row.summary)
    print("\nReference simplified document:\n")
    print(row.simple)
    print("\nPredicted simplified document:\n")
    print(row.prediction)

del biobart_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()